# 多智能体：协调一个专家团队

我们将使用 Claude Managed Agents 和多智能体协调者模式，为一家名为 Northstar 的虚构公司自动化销售提案的撰写。该公司向中型市场的运营团队销售一套工作流自动化平台。

目前，他们的销售代表为每个潜在客户撰写一份定制化提案：研究该潜在客户所在行业通常看重什么，从几百份案例中挑出两个相关案例，依据内部规则表测算定价，并把它组装成一份两页文档。每一步都依赖不同的来源和不同种类的判断。

我们将让一个协调者智能体驱动三个专家来完成这件事。研究员用网络搜索找出该潜在客户所在行业的常见情况。馆员阅读案例库并挑出最匹配的两份。定价建模者只能看到规则文件和席位数。协调者安排它们的先后顺序并撰写提案。

## 1. 设置客户端

首先，安装 SDK 并设置 Anthropic 客户端。multiagent 配置和事件类型属于 Managed Agents 测试版的一部分。


In [ ]:
%%capture
%pip install -qU "anthropic>=0.121.0" python-dotenv

In [ ]:
import os

import anthropic
from dotenv import load_dotenv

load_dotenv()

BETAS = ["managed-agents-2026-04-01"]
MODEL = os.environ.get("COOKBOOK_MODEL", "claude-opus-5")
ADVISOR_MODEL = os.environ.get("COOKBOOK_ADVISOR_MODEL", "claude-opus-5")
client = anthropic.Anthropic()

## 2. 定义三个专家子智能体

接下来，我们创建三位队员。每个队员都有自己的系统提示词、自己的输出形态，以及只完成其工作所需的工具。研究员获得网络搜索，案例挑选者只能读取本地案例库，定价建模者只能看到 `pricing_rules.md` 和席位数。按角色限定工具范围，可以防止定价者从网上抓取竞争对手的数字，也让完整的案例库不进入协调者的上下文。


In [ ]:
def make_agent(name, description, system, tools):
    a = client.beta.agents.create(
        name=name,
        description=description,
        # Tells Anthropic which cookbook this agent came from. Safe to remove.
        metadata={"anthropic_cookbook": "claude-cookbooks/cma-coordinate-specialist-team"},
        model=MODEL,
        system=system,
        tools=tools,
        betas=BETAS,
    )
    print(f"{name}: {a.id}")
    return a.id


prospect_researcher = make_agent(
    "prospect_researcher",
    "Researches what companies in a given industry segment and size tier typically prioritize.",
    """Given a prospect's industry and size, use web search to find:
- What companies in that segment typically list as strategic priorities
- Recent trends or pressures in that industry
- Common operational pain points at that scale
Return via send_to_parent: {"priorities": [...], "recent_moves": [...], "pain_points": [...], "sources": [...]}""",
    [
        {
            "type": "agent_toolset_20260401",
            "configs": [{"name": "web_search"}, {"name": "web_fetch"}],
        }
    ],
)

case_study_picker = make_agent(
    "case_study_picker",
    "Selects the two most relevant case studies from the library for a given prospect profile.",
    """The case study library is in /mnt/user-data/case_studies/. Each file is one customer story.
You will be given a prospect's industry, size, and top priorities. Read the library, score each study on relevance, and pick the two best matches.
Return via send_to_parent: {"picks": [{"file": ..., "customer": ..., "why_relevant": ...}, ...]}""",
    [{"type": "agent_toolset_20260401"}],
)

pricing_modeler = make_agent(
    "pricing_modeler",
    "Builds two or three pricing options for a prospect based on seat count and expected usage.",
    """Pricing rules are in /mnt/user-data/pricing_rules.md. Given a prospect's estimated seat count and usage tier, build:
- a conservative option (annual commit, lower per-seat)
- a flexible option (monthly, higher per-seat)
- if seat count > 500, an enterprise option with a platform fee
Show the first-year total for each. Return via send_to_parent: {"options": [{"name": ..., "structure": ..., "year_one_total": ...}, ...]}""",
    [{"type": "agent_toolset_20260401"}],
)

## 3. 给团队提供可用的材料

馆员需要有一个可供挑选的案例库。我们提供覆盖医疗、制造、物流、零售、金融科技和公共部门的七份简短案例，这样你可以看到它真正挑出契合我们潜在客户的那两份。


In [ ]:
CASE_STUDIES = [
    {
        "slug": "stclair_health",
        "title": "St. Clair Health",
        "industry": "regional hospital network",
        "employees": 6200,
        "summary": """Challenge: credentialing and prior-auth workflows spread across 11 systems.
Result with Northstar: consolidated to 3 automated workflows; prior-auth turnaround down 58%; $1.9M annual labor savings.""",
    },
    {
        "slug": "blueridge_health_plan",
        "title": "BlueRidge Health Plan",
        "industry": "regional payer",
        "employees": 2800,
        "summary": """Challenge: claims-adjudication exceptions queued in email; 19% required manual rework.
Result with Northstar: exception routing automated end-to-end; rework rate down to 6%; 11-day faster average claim resolution.""",
    },
    {
        "slug": "calder_mfg",
        "title": "Calder Manufacturing",
        "industry": "industrial",
        "employees": 3100,
        "summary": """Challenge: purchase-order approvals averaging 9 days.
Result with Northstar: PO cycle time cut to 2.1 days; 14% reduction in maverick spend.""",
    },
    {
        "slug": "northwind",
        "title": "Northwind Logistics",
        "industry": "3PL",
        "employees": 4400,
        "summary": """Challenge: carrier-onboarding paperwork took 3 weeks per carrier.
Result with Northstar: onboarding down to 4 days; 22% more carriers activated in Q1.""",
    },
    {
        "slug": "harborview_retail",
        "title": "Harborview Retail Group",
        "industry": "specialty retail",
        "employees": 5600,
        "summary": """Challenge: store-level inventory exceptions handled by regional managers over Slack and spreadsheets.
Result with Northstar: exception triage automated across 140 stores; stockout incidents down 31%.""",
    },
    {
        "slug": "aperture_fintech",
        "title": "Aperture Payments",
        "industry": "fintech",
        "employees": 1900,
        "summary": """Challenge: KYC and merchant-onboarding reviews averaging 6 business days.
Result with Northstar: review SLA cut to 36 hours; onboarding throughput up 2.4x with the same team.""",
    },
    {
        "slug": "summit_county",
        "title": "Summit County Government",
        "industry": "public sector",
        "employees": 3700,
        "summary": """Challenge: building-permit applications routed through five departments by paper packet.
Result with Northstar: single digital intake with parallel department review; median permit time 41 to 17 days.""",
    },
]

### 产品与定价材料

我们同时提供产品一页纸说明（协调者撰写“我们如何帮助”这一节时阅读它）以及定价建模者用来构建方案的定价规则文件。


In [ ]:
PRODUCT = """# Northstar Platform — One-Pager
Northstar is a workflow automation platform for mid-market operations teams.
Core capabilities: visual process builder, 200+ SaaS connectors, role-based approvals, SOC 2 Type II.
Typical results: 40-60% reduction in manual ticket handling, 3-week time-to-first-workflow."""

PRICING = """# Pricing Rules (internal)
- Per-seat list: $65/mo (monthly) or $52/mo (annual commit).
- Usage tiers: light = 1.0x, standard = 1.15x, heavy = 1.30x multiplier on per-seat.
- Enterprise (>500 seats): add $48,000/yr platform fee, per-seat drops to $44/mo annual.
- All options include onboarding; enterprise includes a named CSM."""

### 接好协调者并启动会话

现在我们创建一个环境、上传九个文件，并创建带有三位专家 `multiagent` 名册的协调者。每个条目都是一份完整的智能体定义，拥有自己的模型、提示词和工具集，因此你可以按角色混用不同档位的模型。

名册中还包含一个不是智能体的条目：`{"type": "advisor", "model": ...}` 指定了一个更强的模型，协调者可以在轮次中途咨询它。它没有工具，也不能被派生或发消息。协调者调用它的 `advisor` 工具，平台把到此为止的对话展示给该模型，指导意见再回到这一轮中。在这里，它在撰写之前对案例挑选和定价表述做一次合理性检查。完整模式见 [﷐2﷑](CMA_consult_an_advisor.ipynb)。


In [ ]:
env = client.beta.environments.create(
    name="proposal-meridian",
    config={"type": "anthropic_cloud", "networking": {"type": "unrestricted"}},
)

resources = []


def mount(path, content):
    f = client.beta.files.upload(file=(os.path.basename(path), content.encode(), "text/plain"))
    resources.append({"type": "file", "file_id": f.id, "mount_path": path})


for cs in CASE_STUDIES:
    body = f"# {cs['title']} ({cs['industry']}, {cs['employees']:,} employees)\n{cs['summary']}"
    mount(f"/mnt/user-data/case_studies/{cs['slug']}.md", body)
mount("/mnt/user-data/product_one_pager.md", PRODUCT)
mount("/mnt/user-data/pricing_rules.md", PRICING)

coordinator = client.beta.agents.create(
    name="Proposal Writer",
    # Tells Anthropic which cookbook this agent came from. Safe to remove.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-coordinate-specialist-team"},
    model=MODEL,
    system="""You assemble tailored sales proposals.
Given a prospect name and basic profile:
1. Send the prospect's industry and size to prospect_researcher.
2. Send the prospect's industry, size, and (once the researcher reports back) their priorities to case_study_picker.
3. Send the seat count and usage tier to pricing_modeler.
4. Before writing, consult your advisor to check the case-study picks and the pricing framing against the prospect's priorities.
5. Read /mnt/user-data/product_one_pager.md, then write /mnt/session/outputs/proposal.md with sections:
   Executive summary (tied to their priorities), How we help (from the one-pager),
   Proof (the two case studies), Investment (the pricing options), Next steps.
Keep it to two pages.""",
    tools=[{"type": "agent_toolset_20260401"}],
    multiagent={
        "type": "coordinator",
        "agents": [
            prospect_researcher,
            case_study_picker,
            pricing_modeler,
            # a stronger model the coordinator can consult mid-turn
            {"type": "advisor", "model": ADVISOR_MODEL},
        ],
    },
    betas=BETAS,
)

session = client.beta.sessions.create(
    agent={"type": "agent", "id": coordinator.id, "version": coordinator.version},
    environment_id=env.id,
    resources=resources,
    title="Proposal: Meridian Health",
    betas=BETAS,
)
print(f"Session {session.id} ready with {len(resources)} files mounted")

## 4. 开始撰写提案

我们发送潜在客户画像并观察协调者工作。它会并行启动研究员和定价建模者，然后在研究员的发现返回后再运行案例挑选者，因为挑选者需要那些关注点来评估相关性。在撰写之前，它咨询它的顾问，顾问在事件流中表现为一个短生命周期的 `anthropic.advisor` 线程。


In [ ]:
PROSPECT = {
    "name": "Meridian Health",
    "industry": "regional healthcare system",
    "employees": 8500,
    "estimated_seats": 600,
    "usage_tier": "heavy",
}

client.beta.sessions.events.send(
    session.id,
    betas=BETAS,
    events=[
        {
            "type": "user.message",
            "content": [
                {
                    "type": "text",
                    "text": f"Build a proposal for {PROSPECT['name']}, a {PROSPECT['industry']} with "
                    f"~{PROSPECT['employees']} employees. Estimate {PROSPECT['estimated_seats']} seats "
                    f"at {PROSPECT['usage_tier']} usage. Write to /mnt/session/outputs/proposal.md.",
                }
            ],
        }
    ],
)

with client.beta.sessions.events.stream(session.id, betas=BETAS) as stream:
    for ev in stream:
        if ev.type == "session.thread_created":
            print(f"[spawn] {ev.agent_name}")
        elif ev.type == "agent.thread_message_received":
            if ev.from_agent_name == "anthropic.advisor":
                print("[advice] advisor guidance received")
            else:
                print(f"[report] {ev.from_agent_name} returned")
        elif ev.type == "session.status_idle":
            print("[done]")
            break

### 每位队员回传了什么

在看组装好的提案之前，我们先打印三份原始的 `send_to_parent` 载荷以及顾问的指导意见。每个子智能体都在自己的上下文中、只用自己的工具运行，所以三份报告彼此差别很大；顾问那一段是协调者在撰写前要求的合理性检查。


In [ ]:
def text_of(content):
    return "".join(b.text for b in content if b.type == "text")


for ev in client.beta.sessions.events.list(session.id, limit=1000, betas=BETAS):
    if ev.type == "agent.thread_message_received":
        body = text_of(ev.content) or "[redacted]"  # advisor content can be withheld
        label = (
            "advisor guidance"
            if ev.from_agent_name == "anthropic.advisor"
            else f"send_to_parent from {ev.from_agent_name}"
        )
        print(f"━━━ {label} ({len(body)} chars) ━━━")
        print(body[:1200] + (f"\n…[{len(body) - 1200} more chars]" if len(body) > 1200 else ""))
        print()

## 5. 阅读提案

最后，我们把组装好的提案取出来。协调者用 `write` 工具把它写入 `proposal.md`，所以我们在日志中找到该事件并查看它生成的各个章节。如果想阅读完整文档，直接打印 `proposal`。

In [ ]:
proposal = ""
for ev in client.beta.sessions.events.list(session.id, limit=1000, betas=BETAS):
    if (
        ev.type == "agent.tool_use"
        and ev.name == "write"
        and ev.input["file_path"].endswith("proposal.md")
    ):
        proposal = ev.input["content"]
        break

# Show the section structure rather than the full proposal.
for line in proposal.splitlines():
    if line.startswith("#"):
        print(line)

## 为什么用三个子智能体而不是一个

一个拥有全部三种工具的智能体也能写出这份提案，那为什么要拆分？把每个角色限定在它自己的工具上，意味着定价建模者无法从网上抓取竞争对手的列表价，因为它只有规则文件。案例挑选者在这里读七个文件，但在生产环境里它会读几百个，而这部分体量留在子智能体的上下文里，而不是协调者的上下文里。而且协调者可以决定顺序和交接，自己却不承担任何专家工作。

关于多智能体协调的更多内容，见 [Managed Agents 文档](https://platform.claude.com/docs/en/managed-agents/multi-agent)。
